# BYOL Features for Football Images

This tutorial trains a YOLO backbone with Bootstrap Your Own Latent on the football player detection dataset. BYOL learns from image pixels only and does not require negative pairs. Object labels are read after training only to interpret the learned feature space.

**Audience**

Students who understand basic PyTorch and self-supervised learning.

**Learning goals**

- Understand the online and target networks in BYOL
- Configure a reusable BYOL module
- Train with AMP and two NVIDIA T4 GPUs
- Save and reload the learned backbone
- Extract football image features
- Visualize the features with t-SNE
- Inspect nearest neighbors using cosine similarity


## How BYOL learns

BYOL creates two augmented views of the same image. The online network contains an encoder, projector, and predictor. The target network contains an encoder and projector whose parameters are updated with an exponential moving average.

For prediction $p$ and target $z$, the normalized regression loss is $D(p,z)=2-2(p^Tz)/(||p||||z||)$. The symmetric objective is $L=(D(p_1,sg(z_2))+D(p_2,sg(z_1)))/2$, where $sg$ stops gradients through the target branch.

After each optimizer step, the target parameters follow $xi=m xi+(1-m) theta$. The momentum $m$ gradually increases from 0.996 toward 1.0 during training.


## Notebook roadmap

1. Configure Kaggle
2. Inspect the football dataset
3. Configure BYOL
4. Preview the augmented views
5. Build a reusable BYOL module
6. Train on T4 x2
7. Review the loss and checkpoints
8. Extract validation features
9. Plot t-SNE
10. Inspect nearest neighbors
11. Complete an exercise


## 1. Configure Kaggle

Select GPU T4 x2 in Notebook options, enable Internet access, and attach the iasadpanwhar/football-player-detection-yolov8 dataset through Add Input. The notebook uses mounted files directly and does not call the Kaggle competition API.


In [ ]:
%pip install -q --upgrade "git+https://github.com/rifat963/ssl-detection-lab.git@main" scikit-learn seaborn


In [ ]:
from pathlib import Path
from collections import Counter
from dataclasses import replace
import json
import random

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
import yaml
from PIL import Image
from packaging.version import Version
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2
from tqdm.auto import tqdm
from ultralytics import YOLO

import ssldet
from ssldet import PretrainConfig, available_ssl_modules, launch_distributed_pretrain
from ssldet.backbones import YOLOBackboneEncoder
from ssldet.data import IMAGENET_MEAN, IMAGENET_STD, UnlabeledImageDataset, build_transform
from ssldet.ssl import create_ssl_module

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision("high")
sns.set_theme(style="whitegrid", context="notebook")

assert Version(ssldet.__version__) >= Version("0.5.0")
assert torch.cuda.is_available(), "Select a GPU accelerator before continuing."

GPU_COUNT = torch.cuda.device_count()
DEVICE = torch.device("cuda:0")
GPU_NAMES = [torch.cuda.get_device_name(index) for index in range(GPU_COUNT)]

pd.Series({
    "ssl-detection-lab": ssldet.__version__,
    "PyTorch": torch.__version__,
    "CUDA": torch.version.cuda,
    "GPU count": GPU_COUNT,
    "GPUs": ", ".join(GPU_NAMES),
    "BYOL available": "byol" in available_ssl_modules(),
})


The expected runtime reports two T4 GPUs. BYOL performs four backbone passes for every pair of views: two through the online encoder and two through the target encoder. It is therefore heavier than SimCLR even with AMP enabled.


## 2. Inspect the football dataset


In [ ]:
DATASET_CANDIDATES = [
    Path("/kaggle/input/datasets/iasadpanwhar/football-player-detection-yolov8/football_players_detection/football_players_detection"),
    Path("/kaggle/input/football-player-detection-yolov8/football_players_detection/football_players_detection"),
]

DATASET_ROOT = next((path for path in DATASET_CANDIDATES if path.exists()), None)

if DATASET_ROOT is None:
    mounted_inputs = sorted(str(path) for path in Path("/kaggle/input").glob("*"))
    raise FileNotFoundError(
        "Attach the football-player-detection-yolov8 dataset with Add Input. "
        f"Mounted inputs: {mounted_inputs}"
    )

SPLIT_PATHS = {
    split: {
        "images": DATASET_ROOT / split / "images",
        "labels": DATASET_ROOT / split / "labels",
    }
    for split in ("train", "valid", "test")
}

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def image_files(directory):
    return sorted(
        path for path in directory.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )

dataset_summary = []
for split, paths in SPLIT_PATHS.items():
    images = image_files(paths["images"])
    labels = sorted(paths["labels"].glob("*.txt"))
    dataset_summary.append({"split": split, "images": len(images), "labels": len(labels)})

pd.DataFrame(dataset_summary).set_index("split")


Only train/images is supplied to BYOL. Annotation files remain outside the SSL training pipeline.


In [ ]:
train_images = image_files(SPLIT_PATHS["train"]["images"])
sample_paths = random.Random(SEED).sample(train_images, min(8, len(train_images)))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for axis in axes.flat:
    axis.axis("off")
for axis, path in zip(axes.flat, sample_paths):
    with Image.open(path) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(path.name, fontsize=9)
plt.suptitle("Football training images", fontsize=16)
plt.tight_layout()
plt.show()


## 3. Configure BYOL

The fast tutorial preset uses 160-pixel crops, at most 2,000 training images, and five epochs. Set FAST_RUN to False for the full dataset, 224-pixel crops, and 25 epochs.

The model starts from yolo26n.yaml, so the backbone is randomly initialized rather than loaded from supervised COCO weights.


In [ ]:
FAST_RUN = True
OUTPUT_DIR = Path("/kaggle/working/byol_football")
EPOCHS = 5 if FAST_RUN else 25
IMAGE_SIZE = 160 if FAST_RUN else 224
MAX_IMAGES = 2000 if FAST_RUN else None
PER_GPU_BATCH_SIZE = 32

config = PretrainConfig(
    method="byol",
    image_roots=[str(SPLIT_PATHS["train"]["images"])],
    output_dir=str(OUTPUT_DIR),
    yolo_model="yolo26n.yaml",
    epochs=EPOCHS,
    batch_size=PER_GPU_BATCH_SIZE,
    image_size=IMAGE_SIZE,
    workers=2,
    max_images=MAX_IMAGES,
    seed=SEED,
    learning_rate=3e-4,
    min_learning_rate=3e-6,
    weight_decay=1e-4,
    warmup_epochs=1,
    grad_accum_steps=1,
    gradient_clip=5.0,
    amp=True,
    projection_dim=128,
    hidden_dim=512,
    momentum=0.996,
    final_momentum=1.0,
    save_every=1,
).validate()

pd.Series({
    "preset": "fast tutorial" if FAST_RUN else "full training",
    "model": config.yolo_model,
    "epochs": config.epochs,
    "maximum images": config.max_images or len(train_images),
    "image size": config.image_size,
    "batch per GPU": config.batch_size,
    "world size": GPU_COUNT,
    "starting momentum": config.momentum,
    "final momentum": config.final_momentum,
    "AMP": config.amp,
})


## 4. Preview the augmented views

BYOL receives the same two-view augmentation family used by the contrastive methods, but the other images in the batch are not treated as negatives.


In [ ]:
preview_dataset = UnlabeledImageDataset(train_images, build_transform(config))
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

def display_tensor(tensor):
    return (tensor.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0)

fig, axes = plt.subplots(4, 2, figsize=(8, 14))
for row in range(4):
    first_view, second_view = preview_dataset[row]
    axes[row, 0].imshow(display_tensor(first_view))
    axes[row, 1].imshow(display_tensor(second_view))
    for axis in axes[row]:
        axis.set_xticks([])
        axis.set_yticks([])
axes[0, 0].set_title("Online view")
axes[0, 1].set_title("Target view")
plt.tight_layout()
plt.show()


## 5. Build a reusable BYOL module

The shared trainer creates this module internally. The next cell exposes the same public factory so the online encoder, target encoder, projectors, predictor, and loss can also be reused in a custom PyTorch workflow.


In [ ]:
probe_detector = YOLO(config.yolo_model)
probe_encoder = YOLOBackboneEncoder(probe_detector.model).to(DEVICE)

with torch.inference_mode():
    feature_dim = probe_encoder(
        torch.zeros(1, 3, config.image_size, config.image_size, device=DEVICE)
    ).shape[1]

byol_module = create_ssl_module(
    "byol",
    encoder=probe_encoder,
    feature_dim=feature_dim,
    hidden_dim=config.hidden_dim,
    projection_dim=config.projection_dim,
    momentum=config.momentum,
).to(DEVICE)

first_view, second_view = preview_dataset[0]
with torch.inference_mode(), torch.amp.autocast("cuda"):
    example_loss = byol_module((
        first_view.unsqueeze(0).to(DEVICE),
        second_view.unsqueeze(0).to(DEVICE),
    ))

pd.Series({
    "feature dimensions": feature_dim,
    "projection dimensions": config.projection_dim,
    "example BYOL loss": float(example_loss),
    "target encoder trainable": any(parameter.requires_grad for parameter in byol_module.target_encoder.parameters()),
})


In [ ]:
del byol_module, probe_encoder, probe_detector, first_view, second_view, example_loss
torch.cuda.empty_cache()


The target encoder should report False for trainable parameters. It changes only through the EMA update performed after each optimizer step.


## 6. Train on T4 x2

The launcher writes the configuration and starts one process per visible GPU. AMP uses CUDA float16 autocasting with a modern gradient scaler. The progress bar reports loss, learning rate, and EMA momentum.


In [ ]:
training_result = launch_distributed_pretrain(
    config,
    num_processes=GPU_COUNT,
    config_path=OUTPUT_DIR / "byol_config.yaml",
    check=True,
)

pd.Series({
    "completed": training_result.succeeded,
    "seconds": round(training_result.seconds, 1),
    "output directory": str(training_result.output_dir),
})


The BYOL regression loss commonly decreases as the online predictions align with the slowly moving target representations. Loss alone cannot detect every form of representational collapse, so the feature norm, t-SNE view, and nearest neighbors are checked below.


## 7. Review the loss and checkpoints


In [ ]:
history = pd.read_csv(OUTPUT_DIR / "history.csv")
display(history.round(6))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.lineplot(data=history, x="epoch", y="loss", marker="o", linewidth=2.5, ax=axes[0])
sns.lineplot(data=history, x="epoch", y="ema_momentum", marker="o", linewidth=2.5, ax=axes[1])
axes[0].set_title("BYOL training loss")
axes[0].set_ylabel("Negative cosine loss")
axes[1].set_title("Target-network momentum")
axes[1].set_ylabel("EMA momentum")
for axis in axes:
    axis.set_xlabel("Epoch")
    axis.xaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout()
plt.show()


In [ ]:
manifest = json.loads((OUTPUT_DIR / "run_manifest.json").read_text())
YOLO_CHECKPOINT = Path(manifest["outputs"]["yolo_checkpoint"])
LAST_SSL_CHECKPOINT = Path(manifest["outputs"]["ssl_checkpoint"])
BEST_SSL_CHECKPOINT = OUTPUT_DIR / "best_ssl.pt"

assert YOLO_CHECKPOINT.is_file()
assert LAST_SSL_CHECKPOINT.is_file()
assert BEST_SSL_CHECKPOINT.is_file()

pd.Series({
    "initialization": manifest["initialization"],
    "unlabeled images": manifest["unlabeled_images"],
    "world size": manifest["world_size"],
    "best loss": manifest["best_loss"],
    "YOLO checkpoint": str(YOLO_CHECKPOINT),
    "best BYOL checkpoint for downstream": str(BEST_SSL_CHECKPOINT),
    "last resumable BYOL checkpoint": str(LAST_SSL_CHECKPOINT),
})

The detector-compatible YOLO checkpoint contains the trained online backbone. The full BYOL checkpoints also retain the online and target branches, projector, predictor, optimizer, scheduler, scaler, and training history.

Use `best_ssl.pt` as the input to `byol_yolo26_football_downstream_tutorial.ipynb`. On Kaggle, save this notebook's output and attach it to the downstream notebook as an input.

## 8. Extract validation features

The t-SNE analysis uses the online backbone output before the projector and predictor. Each multi-object image is colored by the rarest annotated class present in that image. Labels are used only for this visualization.


In [ ]:
yaml_candidates = sorted(DATASET_ROOT.parent.rglob("data.yaml"))
dataset_yaml = yaml_candidates[0] if yaml_candidates else None
metadata = yaml.safe_load(dataset_yaml.read_text()) if dataset_yaml else {}
raw_names = metadata.get("names", {})

if isinstance(raw_names, list):
    CLASS_NAMES = {index: name for index, name in enumerate(raw_names)}
elif isinstance(raw_names, dict):
    CLASS_NAMES = {int(index): name for index, name in raw_names.items()}
else:
    CLASS_NAMES = {}

def object_classes(label_path):
    if not label_path.exists():
        return []
    rows = [line.split() for line in label_path.read_text().splitlines() if line.strip()]
    return [int(float(row[0])) for row in rows]

validation_images = image_files(SPLIT_PATHS["valid"]["images"])
validation_class_lists = [
    object_classes(SPLIT_PATHS["valid"]["labels"] / f"{path.stem}.txt")
    for path in validation_images
]
class_frequency = Counter(class_id for values in validation_class_lists for class_id in values)

if not CLASS_NAMES:
    CLASS_NAMES = {class_id: f"class {class_id}" for class_id in sorted(class_frequency)}

pd.DataFrame([
    {
        "class id": class_id,
        "class name": CLASS_NAMES.get(class_id, f"class {class_id}"),
        "objects": count,
    }
    for class_id, count in sorted(class_frequency.items())
]).set_index("class id")


In [ ]:
EMBEDDING_LIMIT = 500
selected_paths = validation_images
if len(selected_paths) > EMBEDDING_LIMIT:
    selected_paths = sorted(random.Random(SEED).sample(selected_paths, EMBEDDING_LIMIT))

evaluation_transform = v2.Compose([
    v2.Resize(config.image_size + 32, antialias=True),
    v2.CenterCrop(config.image_size),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def image_label(path):
    label_path = SPLIT_PATHS["valid"]["labels"] / f"{path.stem}.txt"
    values = set(object_classes(label_path))
    return min(values, key=lambda class_id: class_frequency[class_id]) if values else -1

class FootballFeatureDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = list(paths)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        with Image.open(path) as image:
            tensor = self.transform(image.convert("RGB"))
        return tensor, image_label(path), path.name

feature_dataset = FootballFeatureDataset(selected_paths, evaluation_transform)
feature_loader = DataLoader(
    feature_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

len(feature_dataset)


In [ ]:
detector = YOLO(str(YOLO_CHECKPOINT))
encoder = YOLOBackboneEncoder(detector.model).to(DEVICE).eval()
feature_batches = []
label_batches = []
file_names = []

with torch.inference_mode(), torch.amp.autocast("cuda"):
    for images, labels, names in tqdm(feature_loader, desc="Extracting BYOL features"):
        features = encoder(images.to(DEVICE, non_blocking=True))
        feature_batches.append(F.normalize(features.float(), dim=1).cpu())
        label_batches.append(labels.numpy())
        file_names.extend(names)

feature_matrix = torch.cat(feature_batches)
label_ids = np.concatenate(label_batches)

pd.Series({
    "images": feature_matrix.shape[0],
    "feature dimensions": feature_matrix.shape[1],
    "feature norm mean": feature_matrix.norm(dim=1).mean().item(),
    "mean feature standard deviation": feature_matrix.std(dim=0).mean().item(),
})


The normalized feature norm should be close to one. A non-zero mean feature standard deviation is a basic sign that different images are not all receiving the same representation.


## 9. Plot t-SNE

t-SNE preserves local neighborhoods rather than calibrated global distances. The axes have no physical meaning, and cluster spacing can change with the random seed and perplexity.


In [ ]:
sample_total = len(feature_matrix)
perplexity = min(30.0, max(2.0, (sample_total - 1) / 3))

tsne = TSNE(
    n_components=2,
    perplexity=perplexity,
    learning_rate="auto",
    init="pca",
    max_iter=1000,
    random_state=SEED,
)
coordinates = tsne.fit_transform(feature_matrix.numpy())

plot_frame = pd.DataFrame({
    "t-SNE 1": coordinates[:, 0],
    "t-SNE 2": coordinates[:, 1],
    "class": [CLASS_NAMES.get(int(class_id), "unlabelled") for class_id in label_ids],
    "image": file_names,
})

fig, axis = plt.subplots(figsize=(12, 8))
sns.scatterplot(
    data=plot_frame,
    x="t-SNE 1",
    y="t-SNE 2",
    hue="class",
    palette="tab10",
    s=65,
    alpha=0.82,
    edgecolor="white",
    linewidth=0.35,
    ax=axis,
)
axis.set_title("t-SNE of BYOL football features", fontsize=16)
axis.legend(title="Rarest object present", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


Related football scenes may form local groups even though BYOL never received class labels. Perfect separation is not expected because the label is image-level and a frame can contain players, referees, goalkeepers, and the ball together.


## 10. Inspect nearest neighbors


In [ ]:
QUERY_INDEX = 0
NEIGHBOR_COUNT = min(5, len(feature_matrix) - 1)
similarities = feature_matrix @ feature_matrix[QUERY_INDEX]
neighbor_indices = torch.topk(similarities, k=NEIGHBOR_COUNT + 1).indices.tolist()
neighbor_indices = [index for index in neighbor_indices if index != QUERY_INDEX][:NEIGHBOR_COUNT]
display_indices = [QUERY_INDEX] + neighbor_indices

fig, axes = plt.subplots(1, len(display_indices), figsize=(4 * len(display_indices), 4))
for position, (axis, index) in enumerate(zip(axes, display_indices)):
    with Image.open(selected_paths[index]) as image:
        axis.imshow(image.convert("RGB"))
    class_name = CLASS_NAMES.get(int(label_ids[index]), "unlabelled")
    title = "Query" if position == 0 else f"Similarity {similarities[index]:.3f}"
    axis.set_title(f"{title}\n{class_name}")
    axis.axis("off")
plt.tight_layout()
plt.show()


Nearest neighbors are calculated in the original normalized feature space, so they are a more direct representation check than distances on the t-SNE plot.


## 11. Exercise

Create a second experiment with starting momentum $m=0.99$. Keep the data subset, seed, image size, and number of epochs fixed. Compare its loss curve, mean feature standard deviation, t-SNE neighborhoods, and nearest neighbors with the original run.


In [ ]:
exercise_config = replace(
    config,
    momentum=0.99,
    output_dir="/kaggle/working/byol_football_momentum_099",
).validate()

pd.Series({
    "starting momentum": exercise_config.momentum,
    "final momentum": exercise_config.final_momentum,
    "epochs": exercise_config.epochs,
    "output directory": exercise_config.output_dir,
})


Run the exercise with launch_distributed_pretrain(exercise_config, num_processes=GPU_COUNT), then repeat the feature extraction and visualization cells using its YOLO checkpoint. A lower starting momentum lets the target network follow the online network more quickly early in training.


## Practical checks and downstream hand-off

- Reduce the per-GPU batch from 32 to 16 if CUDA runs out of memory.
- Keep FAST_RUN enabled for a classroom demonstration.
- Low GPU utilization usually indicates that image decoding and two-view augmentation are the bottleneck.
- BYOL is expected to be slower than SimCLR because it evaluates both online and target encoders for both views.
- A decreasing loss does not prove downstream detection improvement.
- Use `best_ssl.pt` in `byol_yolo26_football_downstream_tutorial.ipynb`, then fine-tune with the labelled train/valid splits and evaluate the untouched test split.
- Use a linear probe, k-nearest-neighbor accuracy, or object-detection metrics for quantitative comparison.